In [ ]:
!pip install requests pillow

In [ ]:
import base64
import json
import requests
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
image_path = r"D:\Projects\Computer vision\Sola\invoices\IMG_20260724_201938.jpg"

img = Image.open(image_path)

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
def image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_b64 = image_to_base64(image_path)

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"

payload = {
    "model": "deepseek-ocr:latest",
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": "Extract all text from this receipt exactly as it appears.",
            "images": [image_b64]
        }
    ]
}

response = requests.post(
    OLLAMA_URL,
    json=payload
)

response.raise_for_status()

result = response.json()

In [ ]:
ocr_text = result["message"]["content"]

print(ocr_text)

In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/chat"

prompt = f"""
You are an invoice parser.

Convert the following OCR text into valid JSON.

Rules:
- Return ONLY JSON.
- If a value is missing, use null.
- Do not explain anything.

Schema:

{{
  "vendor": "",
  "invoice_number": "",
  "date": "",
  "time": "",
  "currency": "",
  "subtotal": "",
  "tax": "",
  "discount": "",
  "total": "",
  "payment_method": "",
  "items":[
    {{
      "name":"",
      "quantity":"",
      "unit_price":"",
      "total_price":""
    }}
  ]
}}

OCR TEXT:

{ocr_text}
"""

payload = {
    "model": "gemma3:4b",
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ]
}

response = requests.post(OLLAMA_URL, json=payload)

response.raise_for_status()

formatted = response.json()["message"]["content"]

print(formatted)

In [ ]:
formatted = formatted.replace("```json","").replace("```","").strip()

data = json.loads(formatted)

data

In [ ]:
import pandas as pd

invoice = {
    "Vendor": data["vendor"],
    "Invoice Number": data["invoice_number"],
    "Date": data["date"],
    "Time": data["time"],
    "Currency": data["currency"],
    "Subtotal": data["subtotal"],
    "Tax": data["tax"],
    "Discount": data["discount"],
    "Total": data["total"],
    "Payment": data["payment_method"]
}

df = pd.DataFrame([invoice])

df.to_csv("D:\\Projects\\Computer vision\\SOLA\\Solainvoice.csv", index=False, encoding="utf-8-sig")

df

In [ ]:
items = pd.DataFrame(data["items"])

items

In [ ]:
items.to_csv("D:\\Projects\\Computer vision\\SOLA\\invoice_items.csv", index=False, encoding="utf-8-sig")

In [ ]:
with pd.ExcelWriter("invoice.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Invoice", index=False)
    items.to_excel(writer, sheet_name="Items", index=False)

print("Saved Successfully")